# RSNA Ensemble Map: Models, Scores & Runtime

**Check what a recipe adds before your next ensemble.** Notebook names and scores do not identify its weights. This CPU guide audits eight public recipes, maps their known main checkpoints and provides a reusable source fingerprint tool.

Three useful findings:

- **Helios and Versia share all five nonempty executable cells** after trimming edge whitespace. Their explanations differ.
- The **384-trained Raptor checkpoint is inferred at 320 pixels** in the audited mirror recipe. Training resolution and model input are different facts.
- The extra-224 recipe adds **one checkpoint within the same CoAtNet family**, at **17.5% of the final successful-row blend**.

Use the [score/version table](#1-a-score-belongs-to-a-version), [checkpoint map](#2-the-main-checkpoint-map), [new-weight check](#3-what-does-a-second-recipe-add), or [fingerprint tool](#4-a-fingerprint-you-can-reuse). Everything runs offline on CPU; no MRI, weights or inference are loaded.

In [1]:
import ast, hashlib, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option("display.max_colwidth", 85)

In [2]:
# Frozen first-party source/score snapshots; full data are exported below.
CATALOG = {'snapshot_utc': '2026-10-13T17:40:19.317868+00:00', 'scope': '8 selected public inference notebooks; main prediction weights only', 'recipes': [{'alias': 'baseline', 'ref': 'pilkwang/rsna-knee-baseline-v1', 'current_version': 15, 'source_code_matches_provider': True, 'current_session': 340906482, 'source_file_sha256': 'b32a9155fcc73c519e75e78c08e07d30393efc591c2798a3e40ca92f39832bb8', 'fingerprint': {'code_sha256': 'eae92cc48c5de02c4420af29e6cf6b9547bfeb60aa646db5cba97d10bc0b3984', 'python_ast_sha256': '827f6d63277f8f386269c9beb780d859d27e39709fe96a04be6c3723b19502a6', 'settings_sha256': 'e6acf82ccd01304f65af932424019caaead1fb9d936a56dafc2c2a2a1ab2b29e', 'code_cells': 21, 'python_minor': '3.12'}, 'best_score': '0.891', 'best_score_version': 15, 'best_score_session': 340906482, 'best_score_code_matches_current': True, 'best_score_code_sha256': 'eae92cc48c5de02c4420af29e6cf6b9547bfeb60aa646db5cba97d10bc0b3984', 'public_runtime_seconds': 384.7295143, 'runtime_session': 340906482, 'runtime_status': 'COMPLETE', 'total_votes': 721, 'views': 15703, 'inputs': ['pilkwang/pilkwang-public-dataset-for-notebooks-figures', 'pilkwang/rsna-knee-llm-labels', 'pilkwang/rsna-knee-weights'], 'models': ['metaresearch/dinov2/PyTorch/small/1'], 'gpu': True, 'hardware': 'NvidiaTeslaT4', 'recorded_utc': '2026-10-13T17:39:57.794917+00:00', 'name': 'Pilkwang baseline', 'main_weights': None, 'audit_note': 'DINOv2 pipeline; manifest-selected members depend on the time budget. Main weights not fully enumerated here.', 'url': 'https://www.kaggle.com/code/pilkwang/rsna-knee-baseline-v1?scriptVersionId=340906482'}, {'alias': 'nartaa384', 'ref': 'nartaa/rsna-knee-0949-anatomical-mirror', 'current_version': 2, 'source_code_matches_provider': True, 'current_session': 356175397, 'source_file_sha256': '8f1f96530b08543bd6caeed48489eea7c01657d6994f22e21a8400047fd4e100', 'fingerprint': {'code_sha256': '44b5a16e26eb060bcee59d89c84e353ab881ad5fef0a8bbd94fe204d787c93f4', 'python_ast_sha256': '0bb4217eb5458cf362aa412ca13c0f0f18bab14afd02931ce427b863bf7712fd', 'settings_sha256': '6826cce92325a7dce7869cfa3f91999ea5ad828437d132d122d6850e3cec3267', 'code_cells': 2, 'python_minor': '3.12'}, 'best_score': '0.949', 'best_score_version': 2, 'best_score_session': 356175397, 'best_score_code_matches_current': True, 'best_score_code_sha256': '44b5a16e26eb060bcee59d89c84e353ab881ad5fef0a8bbd94fe204d787c93f4', 'public_runtime_seconds': 65.207586, 'runtime_session': 356175397, 'runtime_status': 'COMPLETE', 'total_votes': 87, 'views': 1281, 'inputs': ['nartaa/rsna-knee-publication-swa-weights-20261007'], 'models': [], 'gpu': True, 'hardware': 'NvidiaTeslaT4', 'recorded_utc': '2026-10-13T17:40:00.445691+00:00', 'name': 'nartaa 384 mirror', 'main_weights': ['R384'], 'audit_note': 'The 384-trained checkpoint is evaluated with INPUT_RES=320 and mirror views.', 'url': 'https://www.kaggle.com/code/nartaa/rsna-knee-0949-anatomical-mirror?scriptVersionId=356175397'}, {'alias': 'nartaa224', 'ref': 'nartaa/rsna-knee-0945-efficient-224crop', 'current_version': 2, 'source_code_matches_provider': True, 'current_session': 356179962, 'source_file_sha256': 'b80d3b72a1c34f2b39b3ea0d2952f039b8d27c3feb5c4db0a7d0ab62ae75354e', 'fingerprint': {'code_sha256': '92680c8e04be8623ca61b47b19c6eee42d6a000e1d7f29847b26fa2a6f54f2d8', 'python_ast_sha256': '60e74354294c3b44b7e11a14bb6aab721d1e03d5ec08afdf5f03b0bd098922af', 'settings_sha256': 'd36d386962a58d1cab7de7919d32897aadabee7b81eff4a039f286cd6c16cf27', 'code_cells': 2, 'python_minor': '3.12'}, 'best_score': '0.945', 'best_score_version': 2, 'best_score_session': 356179962, 'best_score_code_matches_current': True, 'best_score_code_sha256': '92680c8e04be8623ca61b47b19c6eee42d6a000e1d7f29847b26fa2a6f54f2d8', 'public_runtime_seconds': 53.8275184, 'runtime_session': 356179962, 'runtime_status': 'COMPLETE', 'total_votes': 20, 'views': 396, 'inputs': ['nartaa/rsna-knee-publication-swa-weights-20261007'], 'models': [], 'gpu': True, 'hardware': 'NvidiaTeslaT4', 'recorded_utc': '2026-10-13T17:40:03.285961+00:00', 'name': 'nartaa 224 crop', 'main_weights': ['R224'], 'audit_note': 'Different checkpoint and crop within the same CoAtNet family.', 'url': 'https://www.kaggle.com/code/nartaa/rsna-knee-0945-efficient-224crop?scriptVersionId=356179962'}, {'alias': 'r7', 'ref': 'heliosli/0-957-latest-dataset-score-in-one-click', 'current_version': 1, 'source_code_matches_provider': True, 'current_session': 357456820, 'source_file_sha256': 'cb2a0f57962ca9b6668661e34461aad772ed9ed10d576d32772e7c2898f2e315', 'fingerprint': {'code_sha256': 'fda2522c9f6d0799c01541f1cdf784801e84890fdfac7b2635f7adfd9f65a84d', 'python_ast_sha256': None, 'settings_sha256': '8f046627fba1eabdfc0a2bff1d96508a1836a1dd7e082ca5855770bac310e3af', 'code_cells': 5, 'python_minor': '3.12'}, 'best_score': '0.956', 'best_score_version': 1, 'best_score_session': 357456820, 'best_score_code_matches_current': True, 'best_score_code_sha256': 'fda2522c9f6d0799c01541f1cdf784801e84890fdfac7b2635f7adfd9f65a84d', 'public_runtime_seconds': 409.8633694, 'runtime_session': 357456820, 'runtime_status': 'COMPLETE', 'total_votes': 192, 'views': 1158, 'inputs': ['heliosli/rsna-knee-compact-r7cnxfsss', 'nartaa/rsna-knee-publication-swa-weights-20261007', 'heliosli/rsna-orthofoundation-fold0-epoch7-20261004'], 'models': [], 'gpu': True, 'hardware': 'NvidiaTeslaT4', 'recorded_utc': '2026-10-13T17:40:07.350239+00:00', 'name': 'Helios R7 + Raptor', 'main_weights': ['R384', 'R7f0', 'R7f1', 'R7f2', 'R7f3'], 'audit_note': 'One Raptor checkpoint plus four R7 fold weights; auxiliary networks/stackers excluded from map.', 'url': 'https://www.kaggle.com/code/heliosli/0-957-latest-dataset-score-in-one-click?scriptVersionId=357456820'}, {'alias': 'versia', 'ref': 'evgendvorkin/rsna-versia-7', 'current_version': 22, 'source_code_matches_provider': True, 'current_session': 358019432, 'source_file_sha256': '1b2282b6e5bfe83f77eb07e09b830d5ba1033733bb483487f6843abcff685bc2', 'fingerprint': {'code_sha256': 'fda2522c9f6d0799c01541f1cdf784801e84890fdfac7b2635f7adfd9f65a84d', 'python_ast_sha256': None, 'settings_sha256': '810d8d5e4a3a08cb0bf76b9099245f3d088c3b9d6255c4d258124b4fb8075b5d', 'code_cells': 5, 'python_minor': '3.12'}, 'best_score': '0.956', 'best_score_version': 22, 'best_score_session': 358019432, 'best_score_code_matches_current': True, 'best_score_code_sha256': 'fda2522c9f6d0799c01541f1cdf784801e84890fdfac7b2635f7adfd9f65a84d', 'public_runtime_seconds': 446.7016423, 'runtime_session': 358019432, 'runtime_status': 'COMPLETE', 'total_votes': 380, 'views': 7970, 'inputs': ['heliosli/rsna-knee-compact-r7cnxfsss', 'nartaa/rsna-knee-publication-swa-weights-20261007', 'heliosli/rsna-orthofoundation-fold0-epoch7-20261004'], 'models': [], 'gpu': True, 'hardware': 'NvidiaTeslaT4', 'recorded_utc': '2026-10-13T17:40:11.266846+00:00', 'name': 'Versia 7', 'main_weights': ['R384', 'R7f0', 'R7f1', 'R7f2', 'R7f3'], 'audit_note': 'All five nonempty code cells match Helios after edge whitespace trimming.', 'url': 'https://www.kaggle.com/code/evgendvorkin/rsna-versia-7?scriptVersionId=358019432'}, {'alias': 'extra224', 'ref': 'varshithakolupuri/rsna-knee-one-unused-checkpoint-lifts-the-0-956', 'current_version': 1, 'source_code_matches_provider': True, 'current_session': 357929858, 'source_file_sha256': '6406e92300011e48f0534ce74a91f5b7a0629b3771168ab7275f5584f884f36d', 'fingerprint': {'code_sha256': '1f637931a17825ba42dd67857a90854a3b7a6bb7d0b1022e0f3acada0a342a13', 'python_ast_sha256': None, 'settings_sha256': '8f046627fba1eabdfc0a2bff1d96508a1836a1dd7e082ca5855770bac310e3af', 'code_cells': 7, 'python_minor': '3.12'}, 'best_score': '0.956', 'best_score_version': 1, 'best_score_session': 357929858, 'best_score_code_matches_current': True, 'best_score_code_sha256': '1f637931a17825ba42dd67857a90854a3b7a6bb7d0b1022e0f3acada0a342a13', 'public_runtime_seconds': 457.2314892, 'runtime_session': 357929858, 'runtime_status': 'COMPLETE', 'total_votes': 23, 'views': 183, 'inputs': ['heliosli/rsna-knee-compact-r7cnxfsss', 'nartaa/rsna-knee-publication-swa-weights-20261007', 'heliosli/rsna-orthofoundation-fold0-epoch7-20261004'], 'models': [], 'gpu': True, 'hardware': 'NvidiaTeslaT4', 'recorded_utc': '2026-10-13T17:40:14.279239+00:00', 'name': 'Varshitha +224', 'main_weights': ['R384', 'R224', 'R7f0', 'R7f1', 'R7f2', 'R7f3'], 'audit_note': 'Adds R224 at 17.5% of the final successful-row rank blend.', 'url': 'https://www.kaggle.com/code/varshithakolupuri/rsna-knee-one-unused-checkpoint-lifts-the-0-956?scriptVersionId=357929858'}, {'alias': 'mavit', 'ref': 'hengck23/mavit-288-baseline-01-1-5sec-study-1gpu', 'current_version': 4, 'source_code_matches_provider': True, 'current_session': 356295837, 'source_file_sha256': '41de28b37abcc1caba51f92a4e7117579ad262b240dc52f881cb5fe25f92d59f', 'fingerprint': {'code_sha256': 'f362f2f5927a5228b0d7e4acbb1f8b372ba4efa9576289ab8b4e23b432bfb67e', 'python_ast_sha256': None, 'settings_sha256': '9db08e5cbdfa221c9da26cc13209e62d645fac39cf159dc756d40a0994cff4cd', 'code_cells': 4, 'python_minor': '3.12'}, 'best_score': '0.935', 'best_score_version': 4, 'best_score_session': 356295837, 'best_score_code_matches_current': True, 'best_score_code_sha256': 'f362f2f5927a5228b0d7e4acbb1f8b372ba4efa9576289ab8b4e23b432bfb67e', 'public_runtime_seconds': 60.0257268, 'runtime_session': 356295837, 'runtime_status': 'COMPLETE', 'total_votes': 31, 'views': 715, 'inputs': ['hengck23/hengck23-rnsa-knee'], 'models': [], 'gpu': True, 'hardware': 'NvidiaTeslaT4', 'recorded_utc': '2026-10-13T17:40:16.402020+00:00', 'name': 'MaViT baseline', 'main_weights': None, 'audit_note': 'External model.py and 00000008.pth. No content SHA in inspected notebook; locator-level evidence only.', 'url': 'https://www.kaggle.com/code/hengck23/mavit-288-baseline-01-1-5sec-study-1gpu?scriptVersionId=356295837'}, {'alias': 'hot_leader', 'ref': 'degnonguidi/rsna-knee-abnormality-detection', 'current_version': 10, 'source_code_matches_provider': True, 'current_session': 357933579, 'source_file_sha256': '7d98797592dff6a9faa32951f129465714f44d8d708cf1954accf3c01f33fed8', 'fingerprint': {'code_sha256': '2ce0954a0b04fac6e543240419ae3757330c148a3c9a363fe85a5d3287af2e69', 'python_ast_sha256': None, 'settings_sha256': '28625dd97edc865ab0efd6a02d6d2662c5dc5ae5a4b20a31e41fc5da83f9f778', 'code_cells': 5, 'python_minor': '3.12'}, 'best_score': '0.956', 'best_score_version': 10, 'best_score_session': 357933579, 'best_score_code_matches_current': True, 'best_score_code_sha256': '2ce0954a0b04fac6e543240419ae3757330c148a3c9a363fe85a5d3287af2e69', 'public_runtime_seconds': 443.7885014, 'runtime_session': 357933579, 'runtime_status': 'COMPLETE', 'total_votes': 37, 'views': 290, 'inputs': ['heliosli/rsna-knee-compact-r7cnxfsss', 'nartaa/rsna-knee-publication-swa-weights-20261007', 'heliosli/rsna-orthofoundation-fold0-epoch7-20261004'], 'models': [], 'gpu': True, 'hardware': 'NvidiaTeslaT4', 'recorded_utc': '2026-10-13T17:40:19.317868+00:00', 'name': 'Degnonguidi dual-engine', 'main_weights': ['R384', 'R7f0', 'R7f1', 'R7f2', 'R7f3'], 'audit_note': 'Source contains R384 SHA and pins R7 bundle; wrapper uses IPython file-writing magics.', 'url': 'https://www.kaggle.com/code/degnonguidi/rsna-knee-abnormality-detection?scriptVersionId=357933579'}], 'components': [{'key': 'R384', 'label': 'Raptor 384-trained', 'sha256': '7e5315dad125b99fc65b340b3de41de628e9be51ff5835355dd61c86472244ef', 'family': 'CoAtNet / MIL', 'file': 'raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa.pt', 'identity_evidence': 'SHA256 asserted by inspected source', 'model_input': '320 center crop from native 384 field'}, {'key': 'R224', 'label': 'Raptor 224-crop', 'sha256': '3394fdd7e668286956e85af2554bef0386df5cd79d1c1af223b095c828b815d6', 'family': 'CoAtNet / MIL', 'file': 'raptor_ft_alldata_t16_blendjev_oai_d96_c08_r224_swa.pt', 'identity_evidence': 'SHA256 asserted by inspected source', 'model_input': '0.8 center crop, resized to 224'}, {'key': 'R7f0', 'label': 'R7 fold 0', 'sha256': 'ca8a76e024828793e0663e5a2a13461de17e820bed6789eafe3f155072d3612f', 'family': 'ConvNeXt-small', 'file': 'E_lrprior_s0_f0.pt', 'identity_evidence': 'R7 bundle manifest; inspected source pins bundle 8c7b666e27dc0e8f', 'model_input': '352 crop'}, {'key': 'R7f1', 'label': 'R7 fold 1', 'sha256': 'fb432cf544f218608dbaecf7a8dd3c8b6a26b34a3dfe2dc198172c3e222d5910', 'family': 'ConvNeXt-small', 'file': 'E_lrprior_s0_f1.pt', 'identity_evidence': 'R7 bundle manifest; inspected source pins bundle 8c7b666e27dc0e8f', 'model_input': '352 crop'}, {'key': 'R7f2', 'label': 'R7 fold 2', 'sha256': 'a9718ae111d6307492e724eb1767f0b5a77397d9a4d82c50c1e146b046523bf7', 'family': 'ConvNeXt-small', 'file': 'E_lrprior_s0_f2.pt', 'identity_evidence': 'R7 bundle manifest; inspected source pins bundle 8c7b666e27dc0e8f', 'model_input': '352 crop'}, {'key': 'R7f3', 'label': 'R7 fold 3', 'sha256': '073c22c71a9946c1457e37e56fec145f073a921da106b55a10884da01a40fc94', 'family': 'ConvNeXt-small', 'file': 'E_lrprior_s0_f3.pt', 'identity_evidence': 'R7 bundle manifest; inspected source pins bundle 8c7b666e27dc0e8f', 'model_input': '352 crop'}], 'sources': {'r7_manifest_sha256': 'c4bb0f7399ada9a020e0c15cae6252033cf0c4335e999961793a176063074ec6', 'all_scores': 'First-party author notebook score/version snapshots; not our CV or reproduction', 'weights': 'Source-declared checkpoint SHA values and saved R7 manifest; model bytes not downloaded in this task'}}
recipes = CATALOG['recipes']
by_alias = {r['alias']: r for r in recipes}
print('Snapshot:', CATALOG['snapshot_utc'])

## 1. A score belongs to a version

Each score below comes from the original author's notebook score record, bound to its scored source. The code-match column compares the current source with the source of that score. **It does not reproduce the score**, prove training exclusions, or guarantee current external inputs are unchanged.

Runtime is the **public run total reported by Kaggle**, including setup. Study counts, views, pipelines and machines can differ. Use it as context, not a fair speed benchmark or hidden-test forecast. Titles such as “1.5 sec/study” use a different denominator.

This is a selected audit, not a performance leaderboard. Two recipes with the same reported score may still have different underlying scores.

In [3]:
summary = pd.DataFrame([{
    "Recipe": r["name"], "Current version": r["current_version"],
    "Author public score": r["best_score"] or "Unavailable", "Scored version": r["best_score_version"],
    "Current code matches scored code": r["best_score_code_matches_current"],
    "Public run seconds": round(r["public_runtime_seconds"], 1) if r["public_runtime_seconds"] is not None else np.nan,
    "Source": f'<a href="{r["url"]}">version-bound link</a>'
} for r in recipes])
display(summary.style.hide(axis="index"))

## 2. The main checkpoint map

Colored cells mean the inspected recipe configures that **source-declared main weight identity**. Raptor identities come from full SHA256 values asserted in executable source. The four R7 identities come from its bundle manifest; the recipes pin bundle `8c7b666e27dc0e8f`.

No model files were downloaded or hashed by this guide. Auxiliary anatomy/side networks, stackers and calibration artifacts are outside this map. **Pilkwang and MaViT are unresolved here**, so they are excluded from the matrix; a blank row would falsely imply verified disjointness.

Identical weights can be used with different crops, decoders, augmentations and calibration. That may change predictions. Checkpoint overlap is a source fact, not a measurement of error correlation or blend quality.

In [4]:
known = [r for r in recipes if r['main_weights'] is not None]
keys = [c['key'] for c in CATALOG['components']]
matrix = pd.DataFrame([[int(k in r['main_weights']) for k in keys] for r in known],
                      index=[r['name'] for r in known], columns=keys)
fig, ax = plt.subplots(figsize=(8.5, 3.8))
ax.imshow(matrix, cmap='Blues', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(keys)), keys)
ax.set_yticks(range(len(known)), matrix.index)
for i in range(len(known)):
    for j in range(len(keys)):
        ax.text(j, i, '●' if matrix.iloc[i,j] else '—', ha='center', va='center',
                color='white' if matrix.iloc[i,j] else '#73808c', fontsize=14)
ax.set(title='Declared main weights: six resolved recipes, six identities', xlabel='R7f0–R7f3 are four folds, not four model families')
ax.spines[:].set_visible(False)
fig.tight_layout()
plt.show()

In [5]:
components = pd.DataFrame(CATALOG['components'])
display(components[['key','family','model_input','file']].style.hide(axis='index'))
unresolved = pd.DataFrame([{'Recipe': r['name'], 'Why unresolved': r['audit_note']}
                           for r in recipes if r['main_weights'] is None])
display(unresolved.style.hide(axis='index'))

## 3. What does a second recipe add?

Choose two **resolved** recipes. This is set arithmetic over declared main weight identities. It does not recommend a blend or count independent voters.

In the default comparison, the new weight is R224. It is a different checkpoint in the same CoAtNet/MIL family; R7 remains a different backbone family. The final successful-row formula is `0.325 × rank(R384) + 0.175 × rank(R224) + 0.5 × rank(R7)`.

R224's standalone author score is lower than R384's. That alone does not decide whether it improves an ensemble: complementary positive–negative ordering errors on a clean held-out set are what matter.

In [6]:
BASE_RECIPE = 'r7'
CANDIDATE_RECIPE = 'extra224'  # try 'versia', 'nartaa384', or another resolved alias
a, b = by_alias[BASE_RECIPE], by_alias[CANDIDATE_RECIPE]
if a['main_weights'] is None or b['main_weights'] is None:
    raise ValueError('Resolve the actual weight identities before claiming overlap or novelty')
shared = set(a['main_weights']) & set(b['main_weights'])
added = set(b['main_weights']) - set(a['main_weights'])
print('Shared main weights:', ', '.join(sorted(shared)) or 'none in this resolved scope')
print('Added main weights:', ', '.join(sorted(added)) or 'none')
print('New declared checkpoint count:', len(added))
display(Markdown('**An empty added set does not prove identical predictions.** Check preprocessing, weights in the blend, failure masks and calibration too.'))

## 4. A fingerprint you can reuse

This helper does **not execute** the notebook. It records:

- Ordered nonempty code cells, retaining comments, strings and internal whitespace. Markdown, outputs and edge whitespace are excluded.
- A Python syntax-tree fingerprint when all cells parse as Python. Comments/formatting are ignored, while literal values remain; IPython magics make this unavailable.
- Declared inputs, hardware/image and execution settings, separately from the code.

Equal code hashes mean equal inspected code strings under this normalization. Equal syntax trees are a review clue; their serialization is Python-version dependent, so compare them only with the recorded `python_minor` matching. Neither proves identical dependencies, checkpoint bytes, random state, imported modules or predictions. Unpinned or missing metadata remains an evidence gap.

In [7]:
def recipe_fingerprint(notebook, metadata):
    """Separate ordered code text, parsable Python AST, and declared execution settings."""
    import sys
    cells = []
    for cell in notebook.get("cells", []):
        if cell.get("cell_type") == "code":
            source = cell.get("source", "")
            source = "".join(source) if isinstance(source, list) else source
            if source.strip():
                cells.append(source.strip())
    digest = lambda value: hashlib.sha256(json.dumps(value, ensure_ascii=False, separators=(",", ":")).encode()).hexdigest()
    try:
        trees = [ast.dump(ast.parse(source), include_attributes=False) for source in cells]
        ast_sha = digest(trees)
    except SyntaxError:
        ast_sha = None  # Preserve magics in text; never silently drop them for AST comparison.
    keys = ("language", "kernel_type", "enable_gpu", "enable_tpu", "enable_internet",
            "dataset_sources", "kernel_sources", "model_sources", "competition_sources",
            "docker_image", "machine_shape")
    settings = {key: metadata.get(key) for key in keys}
    return {"code_sha256": digest(cells), "python_ast_sha256": ast_sha,
            "settings_sha256": digest(settings), "code_cells": len(cells),
            "python_minor": f"{sys.version_info.major}.{sys.version_info.minor}"}

In [8]:
# A worked example: explain more without changing code; then change one weight.
base = {'cells': [{'cell_type':'code', 'source':'weight = 0.5\n'}]}
explained = {'cells': [{'cell_type':'markdown', 'source':'A useful explanation'},
                       {'cell_type':'code', 'source':['\n','weight = 0.5\n'], 'outputs':[]} ]}
changed = {'cells': [{'cell_type':'code', 'source':'weight = 0.35\n'}]}
results = [recipe_fingerprint(n, {'enable_gpu':False}) for n in [base, explained, changed]]
# Independent expected serialization: one literal code cell, not generated by the helper.
expected = hashlib.sha256(b'["weight = 0.5"]').hexdigest()
assert results[0]['code_sha256'] == results[1]['code_sha256'] == expected
assert results[2]['code_sha256'] != expected
display(pd.DataFrame({'Example':['base','explanation added','weight changed'],
                       'code fingerprint':[r['code_sha256'][:16] for r in results]}))

In [9]:
# Optional: upload/download notebook JSON + kernel-metadata.json, then set paths.
LOCAL_NOTEBOOKS = []  # [('/kaggle/input/my-files/recipe.ipynb', '/kaggle/input/my-files/kernel-metadata.json')]
fingerprint_rows = []
for notebook_path, metadata_path in LOCAL_NOTEBOOKS:
    note = json.loads(Path(notebook_path).read_text())
    metadata = json.loads(Path(metadata_path).read_text())
    fingerprint_rows.append({'path': notebook_path, **recipe_fingerprint(note, metadata)})
if fingerprint_rows:
    display(pd.DataFrame(fingerprint_rows))
else:
    print('No reader files supplied. The snapshot fingerprints are in the downloads below.')

## 5. One verified duplicate-code example

Helios and Versia have different notebook prose and vote histories. In this snapshot, all five nonempty executable cells match after trimming only edge whitespace. The code fingerprint below is computed from separately collected source files, not inferred from their titles or scores.

Different views/votes cannot be attributed to explanation quality from this comparison alone: publication timing, audience and visibility also differ. Source equivalence is not an accusation; credited explanatory forks can be useful.

In [10]:
pair = [by_alias[k] for k in ['r7','versia']]
assert pair[0]['fingerprint']['code_cells'] == pair[1]['fingerprint']['code_cells'] == 5
assert pair[0]['fingerprint']['code_sha256'] == pair[1]['fingerprint']['code_sha256']
display(pd.DataFrame([{'Recipe':r['name'], 'Version':r['current_version'],
    'Code SHA256':r['fingerprint']['code_sha256'][:20],
    'Declared settings SHA256':r['fingerprint']['settings_sha256'][:20]} for r in pair]))

## 6. A practical decision table

| Your question | Evidence to inspect | What you can conclude |
|---|---|---|
| Is this the same source? | Ordered code fingerprints and a diff | Code identity under the stated normalization |
| Does it add a checkpoint? | Model file content SHA and resolved membership | New weight identity; architecture/error independence still unproven |
| Will it run for me? | Public inputs, exact versions, environment and successful logs | Engineering compatibility after your own check |
| Does it improve my ensemble? | Patient-grouped held-out predictions, training exclusions, per-finding AUC and paired uncertainty | Quality evidence on that eligible validation population |
| Does its displayed score reproduce? | Source-bound completed competition receipt | The submitted version's public result |

The smallest useful next step is often to resolve the inputs and weights first. If two recipes share the same main weights, inspect their views and calibration before scheduling another model stage. If they add a new checkpoint, measure complementary errors before choosing its weight.

Known limitation of the extra-224 recipe: the inherited final blend uses the R384 success mask for the R224 ranks and does not independently export a complete R224 per-study failure mask. Process success alone is insufficient to prove every study was scored. The map does not repair that recipe or claim a better score.

In [11]:
output = Path('ensemble_map_downloads')
output.mkdir(exist_ok=True)
summary.drop(columns='Source').to_csv(output/'recipes.csv', index=False)
components.to_csv(output/'checkpoint_identities.csv', index=False)
matrix.to_csv(output/'resolved_checkpoint_map.csv')
(output/'evidence_manifest.json').write_text(json.dumps(CATALOG, indent=2))
print('Downloads:', ', '.join(p.name for p in sorted(output.iterdir())))

## Sources, scope and reader access

All recipe names above link to the inspected public sessions. Original models and inference are credited to Pilkwang Kim, nartaa, Heliosli, Evgen Dvorkin, Varshithakolupuri, hengck23, Degnonguidi and upstream Dread Development/timm contributors where applicable. This guide contributes the fingerprint method, version-bound comparison, main-weight map and evidence exports. It executes none of their modeling code and distributes no model weights, MRI data, reports or study-level labels.

Checkpoint hashes are source/manifest declarations. To verify a downloaded file, compute its content SHA256 and compare it with the full value in `checkpoint_identities.csv`; a filename or an architecture name is insufficient. Public source access succeeded at collection time, but all attached inputs were not downloaded or tested in a reader's account. Model-use and source-data terms still apply.

The score table reports author results, not our CV. It does not validate unrounded gains, clinical performance, hidden runtime, OOF exclusions or ensemble diversity. This guide has no prediction score badge because it does not perform a competition submission.

The reader-first structure follows useful patterns in [Pilkwang's baseline](https://www.kaggle.com/code/pilkwang/rsna-knee-baseline-v1), the [official Efficiency LB](https://www.kaggle.com/code/ryanholbrook/rsna-knee-abnormalities-efficiency-lb), and the classic [data-exploration walkthrough](https://www.kaggle.com/code/pmarcelino/comprehensive-data-exploration-with-python) and [stacking tutorial](https://www.kaggle.com/code/serigne/stacked-regressions-top-4-on-leaderboard): show a concrete question, inspect evidence, then provide a reusable next action. Their popularity does not guarantee engagement with this guide.

[Official evaluation](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/overview/evaluation) · [Notebook documentation](https://www.kaggle.com/docs/notebooks) · [Progression guidance](https://www.kaggle.com/progression/code). Snapshot facts can become stale; refresh the source/version evidence before relying on a newer recipe.